In [1]:
using LatticeDiracOperators
using Gaugefields
using Wilsonloop
using LMP;

[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018] 
┌ Info: Skipping precompilation due to precompilable error. Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
[ Info: Threads backend loaded with 1 threads


In [2]:
NX = 4
NY = 4
NZ = 4
NT = 4
Nwing = 0
Dim = 4
NC = 3

# const pes =  (2, 2, 2, 2)
# const cuda = true
# PEs = pes

U= Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot")
x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(x)
t = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(t)
x=x/sqrt(norm2(x))
t=t/sqrt(norm2(t))

params = Dict()
params["Dirac_operator"] = "Wilson"
params["κ"] = 0.14
params["eps_CG"] = 1.0e-8
params["verbose_level"] = 3

D= Dirac_operator(U,x,params);

In [3]:
Ritz=Ritz_Layer(D);

In [4]:
dl=DenseLinearLayer(2,25,true);

In [5]:
P=Vector{Paths}(undef,9)
for i in 1:9
    if i<=5
        P[i]=Paths([[i-1]])
    else
        P[i]=Paths([[-(i-5)]])
    end
end
dirs = [1, 2, 3, 4, -1, -2, -3, -4]

max_len = 4                  # 만들고 싶은 최대 길이
lengths = Int[]

L = 2                          # 2^0 = 1도 포함하고 싶으면 1부터,
while L <= max_len             # 2부터만이면 L = 2로 시작
    push!(lengths, L)
    L *= 2
end

for d in dirs
    for L in lengths           # L = 1,2,4,8,...(max_len 이하)
        path_vec = fill(d, L)  # [d,d,...,d] (길이 L)
        push!(P, Paths([path_vec]))
    end
end

In [6]:
i=0.1
pl=length(P)
PT=PTLayer(P,U,pl)
PT1=PTLayer(P,U,pl)
PT2=PTLayer(P,U,pl)
PT3=PTLayer(P,U,pl)
PT4=PTLayer(P,U,pl)
PT.W=i*PT.W
PT1.W=i*PT1.W
PT2.W=i*PT2.W
PT3.W=i*PT3.W
PT4.W=i*PT4.W;

In [7]:
fe=fermionfield_Layer(x,2);
Norm=NormLayer();
gram=GramSchmidt_Layer([deepcopy(x)]);

In [8]:
# 테스트할 레이어 구성
layers = [fe, dl, PT3, Norm, Ritz]

# forward propagation (최신 API: input=D, target=D)
u, f = forward_propagation(layers, D, D)
println("f = ", f)

f = 32.80975052980522


In [9]:
# 수치 미분으로 PTLayer(layers[3])의 gradient 계산
epsilon = 1e-8
nW = length(PT3.W)
grad_num = deepcopy(PT3.W)

for i in 1:nW
    for u_idx in 1:4
        for v_idx in 1:4
            local layer2 = deepcopy(layers)
            layer2[3].W[i][u_idx, v_idx] += epsilon
            _, y1 = forward_propagation(layer2, D, D)
            grad_num[i][u_idx, v_idx] = (y1 - f) / epsilon
        end
    end
end
println("수치 미분 완료")

수치 미분 완료


In [ ]:
# 해석적 gradient 계산 — train과 동일한 방식 사용
γ = Clifford()
L = length(layers)

# backward_propagation! 사용 (train과 동일)
# delta는 L-1 길이, delta[i] = ∂f/∂u[i+1]
# train에서: gradient(layers[i], ..., u[i], delta[i], γ)
delta_buf = Vector{Any}(undef, L-1)
for i in 1:L-1
    delta_buf[i] = deepcopy(u[i+1])
end
backward_propagation!(layers, u, 1, delta_buf)

# PTLayer는 layers[3], train에서는 gradient(layers[3], ..., u[3], delta[3], γ)
grad_ana = gradient(layers[3], layers[3].activation, u[3], delta_buf[3], γ)
println("해석적 gradient 완료 (backward_propagation! 사용)")
println("delta_buf[3] type: ", typeof(delta_buf[3]))
println("length(delta_buf[3]): ", length(delta_buf[3]))

해석적 gradient 완료, delta index used = 2


In [ ]:
# gradient 비교: 수치 미분 vs 해석적 gradient
# 수치 미분(real epsilon)은 ∂f/∂Re(W) 를 줌 (실수)
# 해석적 gradient = Matmul!(delta, hop(z)) 는 복소수
# f가 실수함수일 때: ∂f/∂Re(W) = 2*Re(∂f/∂W)  또는  = Re(grad_ana) (Ritz의 2배가 포함됐을 때)
println("=" ^ 60)
println("PTLayer gradient 검증")
println("=" ^ 60)

g = grad_ana[1]  # gradient(PTLayer,...) returns [grad], so grad_ana[1] is Vector{Spin_matrix}

# 먼저 몇 개만 출력해서 패턴 확인
for i in 1:min(3, length(g))
    println("\n--- W[$i] ---")
    println("수치 (∂f/∂Re W): ", real.(grad_num[i].sm[1:2, 1:2]))
    println("해석 (complex):  ", g[i].sm[1:2, 1:2])
    println("Re(해석):        ", real.(g[i].sm[1:2, 1:2]))
    println("2*Re(해석):      ", 2 .* real.(g[i].sm[1:2, 1:2]))
end

# 전체 비교 — 3가지 케이스
max_diff_direct = 0.0
max_diff_real   = 0.0
max_diff_2real  = 0.0
max_scale       = 0.0

for i in 1:length(g)
    for u_idx in 1:4, v_idx in 1:4
        gn = real(grad_num[i][u_idx, v_idx])
        ga = g[i][u_idx, v_idx]
        global max_diff_direct = max(max_diff_direct, abs(gn - ga))
        global max_diff_real   = max(max_diff_real,   abs(gn - real(ga)))
        global max_diff_2real  = max(max_diff_2real,  abs(gn - 2*real(ga)))
        global max_scale       = max(max_scale, abs(gn))
    end
end
println("\n" * "-" ^ 60)
println("max |∂f/∂Re(W)|  = ", max_scale)
println("max diff direct   = ", max_diff_direct, "  (수치 vs 해석 그대로)")
println("max diff Re(g)    = ", max_diff_real,   "  (수치 vs Re(해석))")
println("max diff 2Re(g)   = ", max_diff_2real,  "  (수치 vs 2*Re(해석))")
println()
println("→ 가장 작은 차이가 올바른 convention")


PTLayer gradient 검증: 수치 미분 vs 해석적 gradient
W[1] max abs diff = 0.0397161445678585
W[2] max abs diff = 0.018599366661933115
W[3] max abs diff = 0.01994786420105334
W[4] max abs diff = 0.026267583462834237
W[5] max abs diff = 0.028166343846927566
W[6] max abs diff = 0.04387257924074963
W[7] max abs diff = 0.03263355620796359
W[8] max abs diff = 0.03022603874875
W[9] max abs diff = 0.031257001284902414
W[10] max abs diff = 0.037038384470063884
W[11] max abs diff = 0.0382331690798042
W[12] max abs diff = 0.04754335160316747
W[13] max abs diff = 0.03133676818934932
W[14] max abs diff = 0.03766930999347236
W[15] max abs diff = 0.025932482625835973
W[16] max abs diff = 0.02276044083291151
W[17] max abs diff = 0.032934981854841644
W[18] max abs diff = 0.0204432098733074
W[19] max abs diff = 0.023590123019136
W[20] max abs diff = 0.027733973209293432
W[21] max abs diff = 0.025568204757892732
W[22] max abs diff = 0.04130199390732521
W[23] max abs diff = 0.03487379821773151
W[24] max abs diff = 0

In [ ]:
# 수치 미분도 복소수 방향으로 테스트
epsilon_c = 1e-8
grad_num_im = deepcopy(PT3.W)

for i in 1:min(3, length(PT3.W))  # 처음 3개만 (시간 절약)
    for u_idx in 1:4
        for v_idx in 1:4
            local layer2 = deepcopy(layers)
            layer2[3].W[i][u_idx, v_idx] += im * epsilon_c
            _, y1 = forward_propagation(layer2, D, D)
            grad_num_im[i][u_idx, v_idx] = (y1 - f) / epsilon_c
        end
    end
end

println("=" ^ 60)
println("복소 방향 수치 미분 결과 (Im epsilon)")
println("=" ^ 60)
for i in 1:min(3, length(grad_ana[1]))
    println("\n--- W[$i] ---")
    println("Re(eps) 수치:  ", grad_num[i].sm[1:2, 1:2])
    println("Im(eps) 수치:  ", grad_num_im[i].sm[1:2, 1:2])
    println("해석 gradient: ", grad_ana[1][i].sm[1:2, 1:2])
    # 수치: ∂f/∂Re(W) = Re_grad, ∂f/∂Im(W)= Im_grad (im*eps 방향)
    # 해석: grad = ∂f/∂Re(W) - im*∂f/∂Im(W) 라면
    recon = grad_num[i].sm[1:2, 1:2] .- im .* grad_num_im[i].sm[1:2, 1:2]
    println("재구성 (Re-im*Im): ", recon)
end


fermionfield_Layer gradient 검증
x[1] rel diff = 1.9741396570091336
x[2] rel diff = 2.7457989167543815


LoadError: BoundsError: attempt to access 2-element Vector{Union{LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_accelerator{3, CUDA.CuArray{ComplexF64, 4, CUDA.DeviceMemory}, 4, :cuda, CUDA.CuArray{ComplexF64, 2, CUDA.DeviceMemory}, CUDA.CuArray{ComplexF64, 4, CUDA.DeviceMemory}}, LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}}} at index [3]

In [ ]:
# DenseLinearLayer gradient 검증
# DenseLinearLayer는 layers[2], delta_buf[2] = ∂f/∂u[3]
grad_dl_num = deepcopy(dl.W)
for i in 1:length(dl.W)
    for u_idx in 1:4
        for v_idx in 1:4
            local layer2 = deepcopy(layers)
            layer2[2].W[i][u_idx, v_idx] += epsilon
            _, y1 = forward_propagation(layer2, D, D)
            grad_dl_num[i][u_idx, v_idx] = (y1 - f) / epsilon
        end
    end
end

# gradient(DenseLinearLayer,...) returns `grad` directly (NOT [grad])  
grad_dl_ana = gradient(layers[2], layers[2].activation, u[2], delta_buf[2], γ)

println("\n" * "=" ^ 60)
println("DenseLinearLayer gradient 검증")
println("=" ^ 60)

# 먼저 패턴 확인
for i in 1:min(3, length(grad_dl_ana))
    println("\n--- W[$i] ---")
    println("수치: ", real.(grad_dl_num[i].sm[1:2, 1:2]))
    println("해석: ", grad_dl_ana[i].sm[1:2, 1:2])
    println("Re(해석): ", real.(grad_dl_ana[i].sm[1:2, 1:2]))
end

# 전체 비교
max_d_direct = 0.0
max_d_real   = 0.0
max_d_2real  = 0.0
max_s        = 0.0

for i in 1:length(grad_dl_ana)
    for u_idx in 1:4, v_idx in 1:4
        gn = real(grad_dl_num[i][u_idx, v_idx])
        ga = grad_dl_ana[i][u_idx, v_idx]
        global max_d_direct = max(max_d_direct, abs(gn - ga))
        global max_d_real   = max(max_d_real,   abs(gn - real(ga)))
        global max_d_2real  = max(max_d_2real,  abs(gn - 2*real(ga)))
        global max_s        = max(max_s, abs(gn))
    end
end
println("\n" * "-" ^ 60)
println("max |∂f/∂Re(W)|  = ", max_s)
println("max diff direct   = ", max_d_direct)
println("max diff Re(g)    = ", max_d_real)
println("max diff 2Re(g)   = ", max_d_2real)


LoadError: UndefRefError: access to undefined reference